In [1]:
import numpy as np
import matplotlib.pyplot as plt
from qutip import *
import pymatching
from scipy.sparse import hstack, kron, eye, csc_matrix, block_diag
from pymatching import Matching

sqrt_pi = np.sqrt(np.pi) # to write less later on
sqrt_2 = np.sqrt(2)

The Toric code is the 2D surface code

https://pymatching.readthedocs.io/en/latest/toric-code-example.html

Need to make new repettition code and stabilisers to get them in the correct matrix shape for time dimension added, also need to accound for peroidic boundary conditions(?)

The tutorial uses X stabs and Z errors i tink so i will do it this way round first

In [ ]:
def toric_rep_code(n):
    row, col = zip(*((i, j) for i in range(n) for j in (i, (i+1) % n)))
    data = np.ones(2*n, dtype=np.uint8)
    return csc_matrix((data, (row, col)))

def toric_x_stab(d): #i think d is my L they use in the tutorial
    Hr = toric_rep_code(d)
    H = hstack([kron(Hr, eye(Hr.shape[1])), kron(eye(Hr.shape[0]), Hr.T)]) #adds the matricies like the normal did but no pre defined things in the tutorial
    H.data = H.data % 2
    H.eliminate_zeros()
    return csc_matrix(H)

def toric_logical_x(d):
    H1 = csc_matrix(([1], ([0],[0])), shape=(1,d), dtype=np.uint8)
    H0 = csc_matrix(np.ones((1, d), dtype=np.uint8))
    logicals = block_diag([kron(H1, H0), kron(H0, H1)])
    logicals.data = logicals.data % 2
    logicals.eliminate_zeros()
    return csc_matrix(logicals)

d = 4
Hx = toric_x_stab(d)
logical_x = toric_logical_x(d)
matching = pymatching.Matching.from_check_matrix(Hx, faults_matrix=logical_x)
n_qubits = Hx.shape[1]

In [ ]:
#error generation now z errors bc using x stabs and x and z anti commute
#use similar structure to surface but time added to make toric?

sigma_vals = np.linspace(0.05, 1, 20)

def toric_z_error(Hx, logical_x, sigma_vals, n_trials):
    logical_rate = []

    for sigma in sigma_vals:
        dp_errors = 0

        for _ in range(n_trials):
            ep = np.zeros(Hx.shape[1], dtype=np.uint8)
            sp_vals = np.zeros(Hx.shape[1])

            for p in range(Hx.shape[1]):
                dp = np.random.normal(0, sigma)
                sp = ((dp + sqrt_pi/2) % sqrt_pi) - sqrt_pi/2
                sp_vals[p] = sp
                kp = int(np.round((dp - sp)/sqrt_pi))
                ep[p] = kp % 2

            #weights_p = ((sqrt_pi - np.abs(sp))**2 - sp**2) / (2*sigma**2) 
            weights_p = ((sqrt_pi - np.abs(sp_vals))**2 - sp_vals**2)/(2*sigma**2)

            matching_t = pymatching.Matching.from_check_matrix(Hx, weights=weights_p)

            syndrome_p = (Hx @ ep) % 2
            correction_p = matching_t.decode(syndrome_p)
            residual_p = (ep + correction_p) % 2

            if np.any((residual_p @ logical_x.T) % 2):
                dp_errors += 1

        errors =  dp_errors

        logical_rate.append(errors / (2 * n_trials))
    return logical_rate    

logical_rate_t = toric_z_error(Hx, logical_x, sigma_vals, n_trials)

#hard decoding 

sigma_vals = np.linspace(0.05, 1, 20)

def toric_z_error_hard(Hx, logical_x, sigma_vals, n_trials):
    logical_rate = []

    for sigma in sigma_vals:
        dp_errors = 0

        for _ in range(n_trials):
            ep = np.zeros(Hx.shape[1], dtype=np.uint8)
            sp_vals = np.zeros(Hx.shape[1])

            for p in range(Hx.shape[1]):
                dp = np.random.normal(0, sigma)
                sp = ((dp + sqrt_pi/2) % sqrt_pi) - sqrt_pi/2
                sp_vals[p] = sp
                kp = int(np.round((dp - sp)/sqrt_pi))
                ep[p] = kp % 2

            matching_2 = pymatching.Matching.from_check_matrix(Hx)

            syndrome_p = (Hx @ ep) % 2
            correction_p = matching_2.decode(syndrome_p)
            residual_p = (ep + correction_p) % 2

            if np.any((residual_p @ logical_x.T) % 2):
                dp_errors += 1

        errors =  dp_errors

        logical_rate.append(errors / (2 * n_trials))
    return logical_rate    

logical_rate_hard = toric_z_error_hard(Hx, logical_x, sigma_vals, n_trials)

plt.semilogy(sigma_vals, logical_rate_t, marker='o', label = 'soft decode')
plt.semilogy(sigma_vals, logical_rate_hard, marker='o', label = 'hard decode')
plt.xlabel("sigma")
plt.ylabel("logical error rate")
plt.title('dp toric log')
plt.legend()
plt.show()

Weights in pymatching relate to measurment probability in the aper this uses $\gamma$

$$\gamma_{synd} = log[(\frac{1-q}{q})]$$

$$\gamma_i = log[\frac{Pr(\tilde{s_i}|s_1=+1)}{Pr(\tilde{s_i}|s_1=-1)}]$$

am pretty sure im doing hard syndrome decoding of H dot e 

do i have to add the gammas into the for loop above?

but is this the big noise error i just want gaussian noise error error 

$$\gamma \approx log(\frac{P_{even}}{P_{odd}})$$

$$weights (\gamma) = \frac{(\sqrt{\pi}-|s|)^2 - s^2}{2 \sigma^2}$$

For noise simulation in the 3d toric code, the tutorial uses a random noise and here the gaussian needs to be used

The gaussian and self correction is done for each rep - which is each surface code over time

this is then used to get the cumulative noise as is done in the tutorial

In [ ]:
def toric_only_qubit_noise(H, logical_x, sigma, n_trials, reps):
    num_stabs, num_qubits = H.shape
    num_errors = 0

    matching = Matching(H, repetitions=reps, faults_matrix=logical_x)
     
    for i in range(n_trials):
        #gkp noise ive done before 
        ep_rounds = np.zeros((num_qubits, reps), dtype = np.uint8)

        for r in range(reps):
            # generates gaussian noise and does self correction with the boundary thing
            dp = np.random.normal(0, sigma, size=num_qubits)
            sp = ((dp + sqrt_pi/2) % sqrt_pi) - sqrt_pi/2
            kp = np.round((dp - sp)/sqrt_pi).astype(np.int8)
            ep_rounds[:, r] = kp % 2

        noise_cumulative = (np.cumsum(ep_rounds, 1) % 2).astype(np.uint8)
        noise_total = noise_cumulative[:, -1]

        syndrome = (H @ noise_cumulative) % 2

        detection_events = np.zeros_like(syndrome, dtype=np.uint8)

        detection_events[:, 0] = syndrome[:, 0]

        detection_events[:, 1:] = (syndrome[:, 1:] + syndrome[:, :-1]) % 2

        predicted_logicals_flips = matching.decode(detection_events.ravel())

        actual_logicals_flipped = (noise_total @ logical_x.T) % 2

        if not np.array_equal(predicted_logicals_flips, actual_logicals_flipped):
            num_errors += 1

    return num_errors / n_trials

logical_rates_t2 = []

for sigma in sigma_vals:

    logical_rates_t2.append(
        toric_only_qubit_noise(
            Hx,
            logical_x,
            sigma,
            n_trials=1000,
            reps=5
        )
    )

plt.semilogy(sigma_vals, logical_rates_t2, marker='o')
plt.xlabel("sigma")
plt.ylabel("logical error rate")
plt.title('dp toric log')
plt.show()

In [ ]:
distances = [3, 5, 7]

for d in distances:
    Hx = toric_x_stab(d)
    logical_x = toric_logical_x(d)
    n_qubits = Hx.shape[1]

    print("\nd =", d)
    print("Hx shape =", Hx.shape)
    print("logical_x shape =", logical_x.shape)

    test = (Hx @ logical_x.T).toarray() % 2

    print(test)
    print(test.max())

    logical_rates_t3 = []

    for sigma in sigma_vals:

        logical_rates_t3.append(
            toric_only_qubit_noise(
                Hx,
                logical_x,
                sigma,
                n_trials=1000,
                reps=d
            )
        )

    plt.semilogy(
        sigma_vals,
        logical_rates_t3,
        marker='o',
        label=f'd={d}'
    )

plt.xlabel("sigma")
plt.ylabel("logical error rate")
plt.title('dp toric log')
plt.legend()
plt.show()  